In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os

from crewai import Agent, Task, Crew
from langchain_groq import ChatGroq

print("Imports successful")

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_KEY"
print("API key set")

In [ ]:
researcher = Agent(
    role="Job Description Analyst",
    goal="Analyse job descriptions and extract key requirements, skills, and qualifications",
    backstory="""You are an expert recruiter with 10 years experience. 
    You can read any job description and instantly identify the top skills,
    experience level required, and what makes a strong candidate.""",
    llm="groq/llama-3.3-70b-versatile",
    verbose=True
)

writer = Agent(
    role="Cover Letter Specialist",
    goal="Write compelling personalised cover letters that get candidates interviews",
    backstory="""You are a professional career coach who has helped 500+ candidates 
    land jobs at top tech companies.""",
    llm="groq/llama-3.3-70b-versatile",
    verbose=True
)

print("Agents created: Researcher + Writer")

In [ ]:
# Profile + Tasks:

neha_profile = """
Name: Neha Kanwadiya
College: IIT Bombay, Engineering Physics, 2027
Internships: 
  - GradPipe: Software Engineering Intern — built backend pipelines, 
    GitHub activity analysis, AI-driven developer marketplace
  - ISB Hyderabad: Research Intern — automated metadata extraction 
    across 4000+ profiles, Python text analysis workflows
Projects:
  - AI Notes Generator: React 18, TypeScript, NLP pipeline, semantic tagging
  - Heart Disease Model: Logistic Regression, Random Forest, PCA, SHAP
  - YouTube Clone: React, Django, JWT auth, adaptive streaming
  - SplitNow: React, expense splitting logic, modular components
Skills: React, TypeScript, Node.js, Django, Python, C++, ML, LLMs, RAG, ChromaDB
"""

job_description = """
Software Engineering Intern — Full Stack + AI
Requirements:
- Strong Python and JavaScript skills
- Experience with React or similar frontend framework  
- Understanding of REST APIs and backend development
- Exposure to ML/AI concepts is a plus
- Problem solving and DSA fundamentals
IIT graduates preferred.
"""

research_task = Task(
    description=f"""Analyse this job description carefully:
{job_description}

Extract and list:
1. Top 5 required technical skills
2. Experience level expected
3. What kind of candidate they want
4. Keywords for the cover letter
5. Match score for Neha (out of 10) with reasoning based on:
{neha_profile}""",
    expected_output="""Structured analysis with:
    - Top 5 skills required
    - Candidate profile they want
    - Match score for Neha out of 10 with reasoning
    - Keywords to include in cover letter""",
    agent=researcher
)

write_task = Task(
    description=f"""Using the job analysis, write a professional 
cover letter for Neha Kanwadiya.

Neha's profile:
{neha_profile}

Rules:
- Maximum 3 paragraphs
- Lead with strongest matching skill
- Mention IIT Bombay in first paragraph
- Include specific project or internship matching JD
- End with confident call to action
- Professional but human tone""",
    expected_output="Complete ready-to-send cover letter of 3 paragraphs",
    agent=writer
)

print("Tasks defined")

In [ ]:
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    verbose=True
)

print("Starting crew...\n")
print("="*60)

result = crew.kickoff()

print("\n" + "="*60)
print("FINAL COVER LETTER:")
print("="*60)
print(result)

In [ ]:
# Dynamic function 

def analyse_and_write(job_desc):
    """Paste any job description — get analysis + cover letter"""
    
    research = Task(
        description=f"""Analyse this job description:
{job_desc}

Evaluate fit for this candidate:
{neha_profile}

List top 5 skills, match score out of 10, and keywords.""",
        expected_output="JD analysis with match score and keywords",
        agent=researcher
    )
    
    write = Task(
        description=f"""Write a 3 paragraph cover letter for Neha Kanwadiya 
based on the job analysis.

Candidate profile: {neha_profile}

Be specific, professional, and concise. 
Mention IIT Bombay and her most relevant project.""",
        expected_output="Complete 3 paragraph cover letter",
        agent=writer
    )
    
    crew = Crew(
        agents=[researcher, writer],
        tasks=[research, write],
        verbose=False
    )
    
    return crew.kickoff()

# Test it
test_jd = """
Software Engineer Intern — AI/ML
We are looking for a passionate engineer to join our AI team.
Requirements: Python, React, REST APIs, basic ML knowledge.
Bonus: LLM experience, RAG pipelines, vector databases.
IIT/NIT graduates preferred. Strong DSA fundamentals required.
"""

print("Running multi-agent crew...\n")
output = analyse_and_write(test_jd)
print("\nFINAL OUTPUT:")
print(output)